## Exploratory Data Analysis on Supply Chain Dataset

#### Importing libraries

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

#### Loading data

In [21]:
df = pd.read_csv("../data/shipments_data.txt", sep="|")
df.head()


,order_id,shipment_id,origin_city,destination_city,carrier_name,shipment_mode,planned_ship_date,actual_ship_date,planned_delivery_date,actual_delivery_date,shipment_status,transportation_cost
0,O0001,S0001,Hyderabad,Pune,Delhivery,Rail,2024-02-11,2024-02-11,2024-02-16,2024-02-16 00:00:00,Delivered,1275
1,O0002,S0002,Chennai,Pune,BlueDart,Road,2024-04-26,2024-04-27,2024-04-29,2024-04-29 00:00:00,Delivered,5314
2,O0003,S0003,Delhi,Delhi,Ecom Express,Road,2024-02-24,2024-02-24,2024-03-01,2024-03-01 00:00:00,Delivered,5527
3,O0004,S0004,Mumbai,Bangalore,XpressBees,Road,2024-03-15,2024-03-15,2024-03-17,NaN,Cancelled,5229
4,O0005,S0005,Mumbai,Pune,Ecom Express,Rail,2024-01-16,2024-01-17,2024-01-20,2024-01-22 00:00:00,Delayed,6033


#### Data Checks

In [22]:
df.shape

(200, 12)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   order_id               200 non-null    object
 1   shipment_id            200 non-null    object
 2   origin_city            200 non-null    object
 3   destination_city       200 non-null    object
 4   carrier_name           200 non-null    object
 5   shipment_mode          200 non-null    object
 6   planned_ship_date      200 non-null    object
 7   actual_ship_date       200 non-null    object
 8   planned_delivery_date  200 non-null    object
 9   actual_delivery_date   187 non-null    object
 10  shipment_status        200 non-null    object
 11  transportation_cost    200 non-null    int64 
dtypes: int64(1), object(11)
memory usage: 18.9+ KB


In [24]:
df.isnull().sum()

order_id                  0
shipment_id               0
origin_city               0
destination_city          0
carrier_name              0
shipment_mode             0
planned_ship_date         0
actual_ship_date          0
planned_delivery_date     0
actual_delivery_date     13
shipment_status           0
transportation_cost       0
dtype: int64

#### converting dates column into datetime data type

In [25]:
date_cols = [
    "planned_ship_date",
    "actual_ship_date",
    "planned_delivery_date",
    "actual_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")


In [26]:
df.dtypes


order_id                         object
shipment_id                      object
origin_city                      object
destination_city                 object
carrier_name                     object
shipment_mode                    object
planned_ship_date        datetime64[ns]
actual_ship_date         datetime64[ns]
planned_delivery_date    datetime64[ns]
actual_delivery_date     datetime64[ns]
shipment_status                  object
transportation_cost               int64
dtype: object

#### Ading few extra columns for more better analysis

In [27]:
df["delivery_delay_days"] = (
    df["actual_delivery_date"] - df["planned_delivery_date"]
).dt.days

In [28]:
df["shipment_duration_days"] = (
    df["actual_delivery_date"] - df["actual_ship_date"]
).dt.days

In [29]:
df[["shipment_status", "delivery_delay_days", "shipment_duration_days"]].head()

,shipment_status,delivery_delay_days,shipment_duration_days
0,Delivered,0.0,5.0
1,Delivered,0.0,2.0
2,Delivered,0.0,6.0
3,Cancelled,NaN,NaN
4,Delayed,2.0,5.0


In [30]:
df.loc[df["shipment_status"] == "Cancelled",
       ["delivery_delay_days", "shipment_duration_days"]] = np.nan

#### Shipment data counts

In [31]:
df["shipment_status"].value_counts()

shipment_status
Delivered    121
Delayed       66
Cancelled     13
Name: count, dtype: int64

#### Average delay by city

In [32]:
df.groupby("destination_city")["delivery_delay_days"].mean().sort_values(ascending=False)

destination_city
Pune         0.657143
Mumbai       0.588235
Chennai      0.518519
Bangalore    0.500000
Hyderabad    0.444444
Delhi        0.416667
Name: delivery_delay_days, dtype: float64

#### Cost by shipment mode

In [33]:
df.groupby("shipment_mode")["transportation_cost"].mean()


shipment_mode
Air     4302.484375
Rail    4781.661290
Road    4428.918919
Name: transportation_cost, dtype: float64

#### Saving Cleaned Datasets

In [35]:
df.to_csv("../data/shipments_cleaned.csv", index=False)

In [37]:
import sqlite3
import pandas as pd

# Load cleaned data
df = pd.read_csv("../data/shipments_cleaned.csv")

# Connect to SQLite DB
conn = sqlite3.connect("../database/logistics.db")

# Write data to SQLite table
df.to_sql("shipments", conn, if_exists="replace", index=False)

conn.close()

print("Data successfully loaded into SQLite database")


Data successfully loaded into SQLite database
